In [26]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode


# 定义工具
@tool
def calculator(expression: str) -> str:
    """执行数学计算。输入格式：'数字1 运算符 数字2'（如'25*14'）"""
    try:
        parts = expression.strip().split()
        if len(parts) != 3:
            return "格式错误，请使用：'数字1 运算符 数字'"
        num1, op, num2 = parts
        num1, num2 = float(num1), float(num2)
        if op == "+":
            return str(num1 + num2)
        elif op == "-":
            return str(num1 - num2)
        elif op == "*":
            return str(num1 * num2)
        elif op == "/":
            return str(num1 / num2)
        else:
            return f"不支持的运算符：{op}"
    except:  # noqa: E722
        return "计算错误"

In [27]:
# 定义天气查询工具
@tool
def get_weather(city:str)->str:
    """查询城市天气。输入：城市名称"""
    mock = {
        "南京":"晴朗，15度",
        "天津":"多余，22度",
        "北京":"雨，28度"
    }
    return mock.get(city, '未知城市')


In [28]:
# 创建agent的工具集
tools = [calculator, get_weather]

In [29]:
# 创建模型对象
glm_client = ChatOpenAI(
    api_key="d1de7475090740ebb6c887167a6a8b98.2WUSfYbmgQwYLKF4",
    base_url="https://open.bigmodel.cn/api/paas/v4/",
    model="glm-4.7",
)

# 给模型安装工具
agent_with_tool = glm_client.bind_tools(tools)

In [ ]:
# 创建聊天节点
def chat_bot(state: MessagesState):
    return {"messages": [agent_with_tool.invoke({"messages": [state["messages"]]})]}

In [36]:
# 条件边创建:判断是否需要调用工具
def should_continue(state:MessagesState):
    last_message = state["messages"][-1]
    print(last_message.model_json_schema(indent = 2))
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return tools
    return END

In [37]:
# 构建图
graph = StateGraph(MessagesState)
graph.add_node("chatbot", chat_bot)# 聊天节点
graph.add_node("tools", ToolNode(tools))# 工具节点，包含所有工具

graph.add_edge(START, "chatbot")
graph.add_conditional_edges("chatbot", should_continue, ["tools", END])
graph.add_edge("tools", "chatbot")

In [38]:
# 校验并挂载运行时
app = graph.compile()

In [40]:
response = app.invoke({"messages": [("user", "北京的天气怎么样？它的气温乘以3是多少")]})

ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.